[![ebac_logo-data_science.png](https://raw.githubusercontent.com/FelipeBebas/Resources/refs/heads/main/assets/ebac-banner-reader.png)](https://github.com/FelipeBebas/Curso-EBAC_Profissao_Cientista_de_Dados)

### **Módulo 27** | PCA | Exercício 1

**Aluno:** [Felipe Barros de Francisco](https://www.linkedin.com/in/felipe-barros-de-francisco-124a3a9b/)<br>
**Data:** Fevereiro de 2026.

# PCA - Tarefa 01: *HAR* com PCA

Vamos trabalhar com a base da demonstração feita em aula, mas vamos explorar um pouco melhor como é o desempenho da árvore variando o número de componentes principais.

In [14]:
import pandas as pd

from sklearn.tree import DecisionTreeClassifier

from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import GridSearchCV

import os
import zipfile
import requests

In [15]:
# Definindo a URL do dataset
zip_file_url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/00240/UCI%20HAR%20Dataset.zip'
local_zip_path = '/content/UCI_HAR_Dataset.zip'
extract_dir = '/content/'

# Definindo um local para extração
if os.path.exists(os.path.join(extract_dir, 'UCI HAR Dataset')):
    print("Removing existing 'UCI HAR Dataset' directory...")
    !rm -rf "{os.path.join(extract_dir, 'UCI HAR Dataset')}"

print(f"Downloading {zip_file_url}...")
!wget -q --show-progress -O "{local_zip_path}" "{zip_file_url}"
print("Download complete.")

print(f"Extracting {local_zip_path} to {extract_dir}...")
!unzip -q "{local_zip_path}" -d "{extract_dir}"
print("Extraction complete.")

# Limpar o arquivo
!rm "{local_zip_path}"

# Verificar se o arquivo existe
if os.path.exists(os.path.join(extract_dir, 'UCI HAR Dataset/train/subject_train.txt')):
    print("Dataset successfully extracted and verified.")
else:
    print("Error: Dataset extraction failed to produce expected files.")

Removing existing 'UCI HAR Dataset' directory...
/content/UCI_HAR_Da     [       <=>          ]  58.17M  43.8MB/s    in 1.3s    
Download complete.
Extracting /content/UCI_HAR_Dataset.zip to /content/...
replace /content/__MACOSX/UCI HAR Dataset/._.DS_Store? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
replace /content/__MACOSX/UCI HAR Dataset/._activity_labels.txt? [y]es, [n]o, [A]ll, [N]one, [r]ename: a
error:  invalid response [a]
replace /content/__MACOSX/UCI HAR Dataset/._activity_labels.txt? [y]es, [n]o, [A]ll, [N]one, [r]ename: a
error:  invalid response [a]
replace /content/__MACOSX/UCI HAR Dataset/._activity_labels.txt? [y]es, [n]o, [A]ll, [N]one, [r]ename: a
error:  invalid response [a]
replace /content/__MACOSX/UCI HAR Dataset/._activity_labels.txt? [y]es, [n]o, [A]ll, [N]one, [r]ename: a
error:  invalid response [a]
replace /content/__MACOSX/UCI HAR Dataset/._activity_labels.txt? [y]es, [n]o, [A]ll, [N]one, [r]ename: All
Extraction complete.
Dataset successfully extracted and ve

In [19]:
features = pd.read_csv(filename_features, header=None, names=['nome_var'], sep="#").squeeze("columns")
labels = pd.read_csv(filename_labels, delim_whitespace=True, header=None, names=['cod_label', 'label'])

subject_train = pd.read_csv(filename_subtrain, header=None, names=['subject_id']).squeeze("columns")
X_train = pd.read_csv(filename_xtrain, delim_whitespace=True, header=None, names=features.tolist())
y_train = pd.read_csv(filename_ytrain, header=None, names=['cod_label'])

subject_test = pd.read_csv(filename_subtest, header=None, names=['subject_id']).squeeze("columns")
X_test = pd.read_csv(ffilename_xtest, delim_whitespace=True, header=None, names=features.tolist())
y_test = pd.read_csv(filename_ytest, header=None, names=['cod_label'])

/tmp/ipython-input-3563425445.py:2: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  labels = pd.read_csv(filename_labels, delim_whitespace=True, header=None, names=['cod_label', 'label'])
/tmp/ipython-input-3563425445.py:5: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  X_train = pd.read_csv(filename_xtrain, delim_whitespace=True, header=None, names=features.tolist())
/tmp/ipython-input-3563425445.py:9: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  X_test = pd.read_csv(ffilename_xtest, delim_whitespace=True, header=None, names=features.tolist())


## Árvore de decisão

Rode uma árvore de decisão com todas as variáveis, utilizando o ```ccp_alpha=0.001```. Avalie a acurácia nas bases de treinamento e teste. Avalie o tempo de processamento.

In [21]:
%%time
# Medindo o tempo de processamento
# Criação do classificador de árvore de decisão com ccp_alpha=0.001
dt_full = DecisionTreeClassifier(random_state=1, ccp_alpha=0.001)
# Treinamento do classificador
dt_full.fit(X_train, y_train)

accuracy_train_full = dt_full.score(X_train, y_train)
accuracy_test_full = dt_full.score(X_test, y_test)

print(f'Acurácia na base de treinamento): {accuracy_train_full:.4f}')
print(f'Acurácia na base de teste): {accuracy_test_full:.4f}')


Acurácia na base de treinamento): 0.9758
Acurácia na base de teste): 0.8802
CPU times: user 8.28 s, sys: 12.3 ms, total: 8.29 s
Wall time: 8.59 s


## Árvore com PCA

Faça uma análise de componemtes principais das variáveis originais. Utilize apenas uma componente. Faça uma árvore de decisão com esta componente como variável explicativa.

- Avalie a acurácia nas bases de treinamento e teste
- Avalie o tempo de processamento

In [23]:
%%time

# Aplicar o PCA com 1 componente aos dados
prcmp = PCA(n_components=1).fit(X_train)

# Transformar os dados de treinamento e teste utilizando as componentes principais encontradas pelo PCA
pca_treino = prcmp.transform(X_train)
pca_teste  = prcmp.transform(X_test)

# Imprime a forma dos dados de treinamento e teste após a transformação
print(f'Dimensão da base de treinamento: {pca_treino.shape}')
print(f'Dimensão da base de teste: {pca_teste.shape}')

# Inicializar um classificador de árvore de decisão com ccp_alpha=0.001 e treina-lo com os dados de treinamento já transformados
clf = DecisionTreeClassifier(ccp_alpha=0.001)
clf.fit(pca_treino, y_train)

# Calcula e imprime a acurácia do classificador nos dados de treinamento e teste
print(f'Acurácia na base de treinamento: {clf.score(pca_treino, y_train)}')
print(f'Acurácia na base de teste: {clf.score(pca_teste, y_test)}\n')

Dimensão da base de treinamento: (7352, 1)
Dimensão da base de teste: (2947, 1)
Acurácia na base de treinamento: 0.499727965179543
Acurácia na base de teste: 0.45707499151679676

CPU times: user 689 ms, sys: 2.79 ms, total: 692 ms
Wall time: 1.02 s


## Testando o número de componentes

Com base no código acima, teste a árvore de classificação com pelo menos as seguintes possibilidades de quantidades de componentes: ```[1, 2, 5, 10, 50]```. Avalie para cada uma delas:

- Acurácia nas bases de treino e teste
- Tempo de processamento


In [26]:
%%time

componentes = [1, 2, 5, 10, 50]

for n in componentes:
    # Executar o PCA com o número de componentes atual
    prcomp = PCA(n_components=n).fit(X_train)

    # Transformar os dados de treinamento e teste nos componentes principais
    pca_treino = prcomp.transform(X_train)
    pca_teste  = prcomp.transform(X_test)

    # Imprimir as dimensões dos dados transformados
    print(f'Dimensões da base de treinamento: {pca_treino.shape}')
    print(f'Dimensões da base de teste: {pca_teste.shape}')

    # Criar e treinar o classificador de árvore de decisão
    clf = DecisionTreeClassifier(ccp_alpha=0.001)
    clf.fit(pca_treino, y_train)

    # Avaliar a acurácia da base de treinamento/teste
    print(f'Acurácia na base de treinamento: {clf.score(pca_treino, y_train)}')
    print(f'Acurácia na base de teste: {clf.score(pca_teste, y_test)}\n')

Dimensões da base de treinamento: (7352, 1)
Dimensões da base de teste: (2947, 1)
Acurácia na base de treinamento: 0.499727965179543
Acurácia na base de teste: 0.45707499151679676

Dimensões da base de treinamento: (7352, 2)
Dimensões da base de teste: (2947, 2)
Acurácia na base de treinamento: 0.6127584330794341
Acurácia na base de teste: 0.5846623685103495

Dimensões da base de treinamento: (7352, 5)
Dimensões da base de teste: (2947, 5)
Acurácia na base de treinamento: 0.8460282916213275
Acurácia na base de teste: 0.7885985748218527

Dimensões da base de treinamento: (7352, 10)
Dimensões da base de teste: (2947, 10)
Acurácia na base de treinamento: 0.8926822633297062
Acurácia na base de teste: 0.8242280285035629

Dimensões da base de treinamento: (7352, 50)
Dimensões da base de teste: (2947, 50)
Acurácia na base de treinamento: 0.919341675734494
Acurácia na base de teste: 0.822870715982355

CPU times: user 5.91 s, sys: 18 ms, total: 5.93 s
Wall time: 7.11 s


## Conclua

- O que aconteceu com a acurácia?
- O que aconteceu com o tempo de processamento?

**O que aconteceu com a acurácia?**

Aumentar o número de componentes aumentou a acurácia na base de testes. Com 50 componentes é atingido uma acurácia de 82%.

**O que aconteceu com o tempo de processamento?**

Diminuindo a dimensionalidade e utilizando a técnica de Análise de Componentes Principais (PCA), a mudança mais significante está na relação de tempo de processamento. Á arvore de decisão completa (561 variáveis) leva mais tempo para ser executada, resultando numa acurácia com cerca de 88% na base de teste.
Ao utilizar o PCA com um único componente, o tempo de processamento foi reduzido junto com sua acurácia.